# 第 2 章习题与解答

> 本章习题聚焦 token 估算、special token 机制和 chat_template 扩展。建议先自己思考,再展开答案。

## Exercise 2.1(易)估算混合文本的 token 数

**题目**:给定以下中英混合文本,先手动估算 token 数,再用 tokenizer 验证:

```
"LLM 是 Large Language Model 的缩写,中文叫「大语言模型」。"
```

提示:中文压缩率约 1.5-1.7 chars/token,英文约 3.5-4.5 chars/token。

<details><summary><b>参考答案</b></summary>

**手动估算**:

- 中文部分:「是」「的缩写,中文叫」「大语言模型」共约 15 个中文字符 → $15 / 1.6 \approx 9$ token
- 英文部分:`LLM is Large Language Model` 约 28 字符 → $28 / 4.0 \approx 7$ token
- 标点和引号:约 3-5 token
- 估计总数:约 **19-21 token**

**实际验证**(需加载 tokenizer):

```python
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("/home/minimind/model")
text = "LLM 是 Large Language Model 的缩写,中文叫「大语言模型」。"
ids = tok.encode(text)
print(f"字符数: {len(text)}")
print(f"token 数: {len(ids)}")
print(f"压缩率: {len(text)/len(ids):.2f}")
```

预期输出约 18-22 token,与估算接近。

**关键收获**:手动估算能帮你预估 API 成本和上下文窗口占用,在实际工程中非常有用。

</details>

## Exercise 2.2(中)为什么 `<|im_start|>` 必须是 special token

**题目**:假设我们不小心把 `<|im_start|>` 注册成了**普通 BPE token**(而非 special token),会有什么问题?请从 encode 和 model 两个角度分析。

<details><summary><b>参考答案</b></summary>

**问题 1:encode 不再原子化**

普通 BPE token 的编码和普通文本一样,会经过 ByteLevel 预分词 + BPE merge。这意味着:

- `<|im_start|>system\n` 可能被预分词切分
- `|` 和 `>` 可能因为频率低而被拆开
- 最终 `<|im_start|>` 可能编码成 2-4 个 id,而非 1 个 id

验证实验:

```python
# 假设性地看普通文本的切分
ids = tok.encode("<|not_special|>", add_special_tokens=False)
print(tok.convert_ids_to_tokens(ids))
# 可能输出: ['<', '|', 'not', '_', 'special', '|', '>']  ← 被拆碎了!
```

**问题 2:模型学不到稳定的边界信号**

ChatML 协议依赖 `<|im_start|>` 作为消息边界的**确定性信号**。如果它有时编码成 1 个 id,有时编码成 3 个 id,模型就:

- 无法稳定地学到「这个位置是一段消息的开始」
- 生成时也可能产出不完整的边界标记
- 对话格式会崩溃

**问题 3:decode 无法往返还原**

普通 token 的 decode 依赖 BPE 的字节逆映射,可能丢失或改变字符。special token 的 decode 是**原样替换**,保证 `decode(encode(x)) == x`。

**结论**:special token 的三个保证 —— **原子编码(1 个 id)、稳定信号、往返一致** —— 是 chat 协议可靠性的基础。

</details>

## Exercise 2.3(难)扩展 chat_template 添加新角色

**题目**:minimind 的 chat_template 目前支持 system / user / assistant 三种角色。请修改模板,添加一个 `translator` 角色,使其渲染为:

```
<|im_start|>translator
{content}<|im_end|>
```

并解释为什么不需要新增 special token。

<details><summary><b>参考答案</b></summary>

**关键洞察**:`<|im_start|>` 后面跟的是**普通文本角色名**,不需要为每个角色新建 special token。角色名 `translator` 只是 `<|im_start|>` 和 `\n` 之间的字符串,会被 BPE 正常编码。

**修改方案**:

chat_template 的核心循环是:

```jinja2
{%- for message in messages %}
    {%- if message.role == "user" or (message.role == "system" and not loop.first) %}
        {{- '<|im_start|>' + message.role + '\n' + content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        ...
    {%- endif %}
{%- endfor %}
```

只需在条件中添加 `translator`:

```jinja2
{%- for message in messages %}
    {%- if message.role == "translator" %}
        {{- '<|im_start|>translator\n' + content + '<|im_end|>\n' }}
    {%- elif message.role == "user" or (message.role == "system" and not loop.first) %}
        {{- '<|im_start|>' + message.role + '\n' + content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        ...
    {%- endif %}
{%- endfor %}
```

更简洁的写法 —— 直接把 translator 加入通用分支:

```jinja2
{%- if message.role in ["user", "system", "translator"] %}
    {{- '<|im_start|>' + message.role + '\n' + content + '<|im_end|>\n' }}
{%- endif %}
```

**为什么不需要新增 special token?**

1. `<|im_start|>` 已经是 special token(id=1),它标记「消息开始」
2. 角色名 `translator` 是普通文本,会被 BPE 编码为几个子词 token
3. 模型通过 `<|im_start|>` 知道这是新消息,通过后面的文本识别角色
4. 只有当角色标记需要**不可分割的原子性**时,才需要 special token(比如 `<think>`)

**验证代码**:

```python
# 修改后的模板可以正常渲染
messages = [
    {"role": "user", "content": "Translate: Hello"},
    {"role": "translator", "content": "你好"},
]
# 使用修改后的 tokenizer
rendered = tok.apply_chat_template(messages, tokenize=False)
# 预期输出包含 <|im_start|>translator\n你好<|im_end|>\n
```

**进阶思考**:如果 translator 角色需要特殊的行为(比如生成时必须以特定标记结束),那才可能需要新增 special token。但对于简单的格式区分,利用 `<|im_start|>` 后跟角色名的机制就够了 —— 这正是 ChatML 协议的设计巧妙之处。

</details>